In [3]:
!pip --version

pip 25.2 from C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\pip (python 3.12)



In [2]:
import sys
print(sys.executable)

C:\Users\DrFirst\AppData\Local\anaconda3\python.exe


In [1]:
import os
import numpy as np
import pandas as pd
import folium
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hmmlearn import hmm

target_file = r"C:\Users\DrFirst\Desktop\drone\포항긴거_20250704_020715.csv"

# 1. 데이터 로드 (오직 X, Y만 추출)
df = pd.read_csv(target_file, header=None)

valid_start_idx = -1
check_limit = min(df.shape[1], 5)

# 타임스탬프 위치 찾기
for i in range(check_limit):
    try:
        if float(df.iloc[0, i]) > 500000:
            valid_start_idx = i
            break
    except:
        continue

# 타임스탬프 다음 2개 컬럼을 Longitude(X), Latitude(Y)로 강제 할당
if valid_start_idx != -1 and df.shape[1] >= valid_start_idx + 3:
    df = df.iloc[:, [valid_start_idx+1, valid_start_idx+2]]
else:
    # 못 찾으면 첫 2개 컬럼을 사용
    df = df.iloc[:, [0, 1]]

df.columns = ['Longitude', 'Latitude']

# 결측치 및 문자열 처리
df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna().reset_index(drop=True)

# 2. X, Y 좌표만으로 비행 물리량(Feature) 수학적 도출
R = 6371000.0 # 지구 반지름(m)
lat_rad = np.deg2rad(df['Latitude'])
lon_rad = np.deg2rad(df['Longitude'])

# X, Y 변화량 (미터 단위)
dx = R * lon_rad.diff().fillna(0) * np.cos(lat_rad.mean())
dy = R * lat_rad.diff().fillna(0)

# 진행 방향(가상 Yaw) 계산 (아크탄젠트 사용)
heading_rad = np.arctan2(dy, dx)
df['Pseudo_Yaw'] = np.rad2deg(heading_rad)
df['Pseudo_Yaw'] = df['Pseudo_Yaw'].bfill()

# 각도 튀는 현상(360 -> 0) 보정
yaw_rad_unwrapped = np.unwrap(np.deg2rad(df['Pseudo_Yaw']))
df['Yaw_unwrap'] = np.rad2deg(yaw_rad_unwrapped)

window_size = 30
df['Yaw_diff'] = df['Yaw_unwrap'].diff().fillna(0).abs()
df['Yaw_rate'] = df['Yaw_diff'].rolling(window=window_size, center=True).mean().fillna(0)
df['Yaw_std'] = df['Yaw_unwrap'].rolling(window=window_size, center=True).std().fillna(0)

# 이동 속도 및 원심 가속도 추정
ds = np.sqrt(dx**2 + dy**2)
df['Speed_proxy'] = ds.rolling(window=window_size, center=True).mean().fillna(0)
df['Centripetal_Acc'] = df['Speed_proxy'] * df['Yaw_rate']

feature_cols = ['Yaw_rate', 'Yaw_std', 'Centripetal_Acc']
df[feature_cols] = df[feature_cols].fillna(0)

# 3. HMM Clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

model = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=100, random_state=42)
model.fit(X_scaled)
df['State_Raw'] = model.predict(X_scaled)

# 어느 쪽이 '회전(Turn)'인지 자동 식별
state_yaw_rates = []
for s in range(2):
    state_mean = np.mean(X_scaled[df['State_Raw'] == s], axis=0)
    state_yaw_rates.append(state_mean[feature_cols.index('Yaw_rate')])
turn_label = np.argmax(state_yaw_rates)

df['State'] = df['State_Raw'].apply(lambda x: 1 if x == turn_label else 0)
df['State_Smooth'] = df['State']

df['Segment_Change'] = df['State_Smooth'].diff().fillna(0).abs()
df['Final_Segment_ID'] = df['Segment_Change'].cumsum().astype(int)

# 4. PCA 기반 직선성 정밀 검증
def is_linear_path(lats, lons, linearity_threshold=0.98):
    if len(lats) < 6:
        return False, 0.0
    lat_rad = np.deg2rad(np.mean(lats))
    xs = np.deg2rad(lons) * R * np.cos(lat_rad)
    ys = np.deg2rad(lats) * R
    coords = np.column_stack([xs, ys])
    
    span = np.ptp(coords, axis=0)
    if np.max(span) < 1.0:
        return True, 1.0
        
    pca = PCA(n_components=2)
    pca.fit(coords)
    ratio = pca.explained_variance_ratio_[0]
    return ratio >= linearity_threshold, ratio

df['State_Final'] = df['State_Smooth'].copy()
df['PCA_Ratio'] = 0.0

rotate_seg_ids = df[df['State_Smooth'] == 1]['Final_Segment_ID'].unique()
for rid in rotate_seg_ids:
    seg_mask = (df['Final_Segment_ID'] == rid)
    seg = df[seg_mask]
    lats = seg['Latitude'].values
    lons = seg['Longitude'].values

    gps_is_linear, pca_ratio = is_linear_path(lats, lons)
    df.loc[seg_mask, 'PCA_Ratio'] = pca_ratio

    yaw_change = seg['Yaw_unwrap'].max() - seg['Yaw_unwrap'].min() if len(seg) > 0 else 0.0
    YAW_CHANGE_THRESHOLD = 45.0
    
    if gps_is_linear and yaw_change < YAW_CHANGE_THRESHOLD:
        df.loc[seg_mask, 'State_Final'] = 3
    else:
        df.loc[seg_mask, 'State_Final'] = 2 if gps_is_linear else 1

df.loc[df['State_Smooth'] == 0, 'State_Final'] = 0
df['Segment_Change_Final'] = (df['State_Final'] != df['State_Final'].shift(1)).astype(int)
df['Final_Segment_ID'] = df['Segment_Change_Final'].cumsum().astype(int)

# 5. 체인 병합 알고리즘 (Chain Merging)
seg_order_df = df.groupby('Final_Segment_ID').agg(
    state_final=('State_Final', 'first'),
    first_idx=('Final_Segment_ID', lambda x: x.index.min())
).reset_index().sort_values('first_idx')

ordered_segs = list(zip(seg_order_df['Final_Segment_ID'].tolist(), seg_order_df['state_final'].tolist()))

runs = []
current_run = []
for seg_id, state in ordered_segs:
    if state in (0, 2, 3):
        current_run.append((seg_id, state))
    else:
        if current_run:
            runs.append(current_run)
            current_run = []
        runs.append([(seg_id, state)])
if current_run:
    runs.append(current_run)

merge_ops = {}
for run in runs:
    if len(run) == 1 and run[0][1] == 1:
        continue
    line_positions = [i for i, (_, st) in enumerate(run) if st == 0]
    if not line_positions:
        continue
    target_id = run[line_positions[0]][0]
    for old_id, _ in run:
        if old_id != target_id:
            merge_ops[old_id] = target_id

for old_id, new_id in merge_ops.items():
    df.loc[df['Final_Segment_ID'] == old_id, 'Final_Segment_ID'] = new_id

all_target_ids = set(merge_ops.values())
for tid in all_target_ids:
    df.loc[(df['Final_Segment_ID'] == tid) & (df['State_Final'].isin([2, 3])), 'State_Final'] = 0

alive_seg_ids = sorted(df['Final_Segment_ID'].unique())
id_mapping = {old_id: new_idx for new_idx, old_id in enumerate(alive_seg_ids, start=1)}
df['Final_Segment_ID'] = df['Final_Segment_ID'].map(id_mapping).astype(int)

# 6. Folium 지도 시각화
center_lat, center_lon = df['Latitude'].mean(), df['Longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=16)

all_coords = df[['Latitude', 'Longitude']].values.tolist()
folium.PolyLine(locations=all_coords, color="#000000", weight=5, opacity=0.5).add_to(m)

colors = [
    '#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#FF00FF', '#00FFFF', '#FF8000', '#0080FF', 
    '#FF0080', '#80FF00', '#00FF80', '#FF4040', '#4040FF', '#FFD700', '#ADFF2F', '#FF69B4', 
    '#1E90FF', '#DC143C'
]

for seg_id in sorted(df['Final_Segment_ID'].unique()):
    group = df[df['Final_Segment_ID'] == seg_id]
    seg_state_final = group['State_Final'].iloc[0]

    if seg_state_final == 0:
        color, weight, opacity, dash_array = colors[seg_id % len(colors)], 3, 0.8, None
    elif seg_state_final == 1:
        color, weight, opacity, dash_array = colors[seg_id % len(colors)], 5, 0.9, '5, 5'
    elif seg_state_final == 2:
        color, weight, opacity, dash_array = '#9400D3', 5, 0.9, '10, 5'
    else:
        color, weight, opacity, dash_array = '#FF6600', 5, 0.9, '10, 5'

    indices = group.index.values
    if len(indices) > 0:
        split_locs = np.where(np.diff(indices) > 1)[0] + 1
        for sub_indices in np.split(indices, split_locs):
            if len(sub_indices) < 2: continue
            sub_coords = df.loc[sub_indices, ['Latitude', 'Longitude']].values.tolist()
            folium.PolyLine(
                locations=sub_coords, color=color, weight=weight, 
                opacity=opacity, dash_array=dash_array
            ).add_to(m)

m

In [1]:
import os
import numpy as np
import pandas as pd
import folium
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from hmmlearn import hmm

target_file = r"C:\Users\DrFirst\Desktop\drone\포항긴거_20250704_020715.csv"

column_names = [
    'Time', 'Longitude', 'Latitude', 'Altitude', 
    'Roll', 'Pitch', 'Yaw', 
    'Mag1_X', 'Mag1_Y', 'Mag1_Z', 
    'Mag2_X', 'Mag2_Y', 'Mag2_Z'
]

df = pd.read_csv(target_file, header=None)

valid_start_idx = -1
check_limit = min(df.shape[1], 5)

for i in range(check_limit):
    try:
        sample_val = float(df.iloc[0, i])
        if sample_val > 500000:
            valid_start_idx = i
            break
    except:
        continue

if valid_start_idx != -1:
    end_idx = valid_start_idx + len(column_names)
    if df.shape[1] >= end_idx:
        df = df.iloc[:, valid_start_idx:end_idx]
        df.columns = column_names
    else:
        df = df.iloc[:, valid_start_idx:]
        df.columns = column_names[:df.shape[1]]
else:
    df.columns = column_names[:df.shape[1]]

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')
df = df.dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)

R = 6371000.0
lat_rad = np.deg2rad(df['Latitude'])
lon_rad = np.deg2rad(df['Longitude'])

dx = R * lon_rad.diff().fillna(0) * np.cos(lat_rad.mean())
dy = R * lat_rad.diff().fillna(0)

heading_rad = np.arctan2(dy, dx)
df['Pseudo_Yaw'] = np.rad2deg(heading_rad)
df['Pseudo_Yaw'] = df['Pseudo_Yaw'].bfill()

yaw_rad_unwrapped = np.unwrap(np.deg2rad(df['Pseudo_Yaw']))
df['Yaw_unwrap'] = np.rad2deg(yaw_rad_unwrapped)

window_size = 30
df['Yaw_diff'] = df['Yaw_unwrap'].diff().fillna(0).abs()
df['Yaw_rate'] = df['Yaw_diff'].rolling(window=window_size, center=True).mean().fillna(0)
df['Yaw_std'] = df['Yaw_unwrap'].rolling(window=window_size, center=True).std().fillna(0)

ds = np.sqrt(dx**2 + dy**2)
df['Speed_proxy'] = ds.rolling(window=window_size, center=True).mean().fillna(0)
df['Centripetal_Acc'] = df['Speed_proxy'] * df['Yaw_rate']

feature_cols = ['Yaw_rate', 'Yaw_std', 'Centripetal_Acc']
df[feature_cols] = df[feature_cols].fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[feature_cols])

model = hmm.GaussianHMM(n_components=2, covariance_type="diag", n_iter=100, random_state=42)
model.fit(X_scaled)
df['State_Raw'] = model.predict(X_scaled)

state_yaw_rates = []
for s in range(2):
    state_mean = np.mean(X_scaled[df['State_Raw'] == s], axis=0)
    state_yaw_rates.append(state_mean[feature_cols.index('Yaw_rate')])
turn_label = np.argmax(state_yaw_rates)

df['State'] = df['State_Raw'].apply(lambda x: 1 if x == turn_label else 0)
df['State_Smooth'] = df['State']

df['Segment_Change'] = df['State_Smooth'].diff().fillna(0).abs()
df['Final_Segment_ID'] = df['Segment_Change'].cumsum().astype(int)

def is_linear_path(lats, lons, linearity_threshold=0.98):
    if len(lats) < 6:
        return False, 0.0
    R = 6371000.0
    lat_rad = np.deg2rad(np.mean(lats))
    xs = np.deg2rad(lons) * R * np.cos(lat_rad)
    ys = np.deg2rad(lats) * R
    coords = np.column_stack([xs, ys])
    
    span = np.ptp(coords, axis=0)
    if np.max(span) < 1.0:
        return True, 1.0
        
    pca = PCA(n_components=2)
    pca.fit(coords)
    ratio = pca.explained_variance_ratio_[0]
    return ratio >= linearity_threshold, ratio

df['State_Final'] = df['State_Smooth'].copy()
df['PCA_Ratio'] = 0.0

rotate_seg_ids = df[df['State_Smooth'] == 1]['Final_Segment_ID'].unique()
for rid in rotate_seg_ids:
    seg_mask = (df['Final_Segment_ID'] == rid)
    seg = df[seg_mask]
    lats = seg['Latitude'].values
    lons = seg['Longitude'].values

    gps_is_linear, pca_ratio = is_linear_path(lats, lons)
    df.loc[seg_mask, 'PCA_Ratio'] = pca_ratio

    if len(seg) > 0:
        yaw_change = seg['Yaw_unwrap'].max() - seg['Yaw_unwrap'].min()
    else:
        yaw_change = 0.0

    YAW_CHANGE_THRESHOLD = 45.0
    
    if gps_is_linear and yaw_change < YAW_CHANGE_THRESHOLD:
        df.loc[seg_mask, 'State_Final'] = 3
    else:
        if gps_is_linear:
            df.loc[seg_mask, 'State_Final'] = 2
        else:
            df.loc[seg_mask, 'State_Final'] = 1

df.loc[df['State_Smooth'] == 0, 'State_Final'] = 0
df['Segment_Change_Final'] = (df['State_Final'] != df['State_Final'].shift(1)).astype(int)
df['Final_Segment_ID'] = df['Segment_Change_Final'].cumsum().astype(int)

seg_order_df = df.groupby('Final_Segment_ID').agg(
    state_final=('State_Final', 'first'),
    first_idx=('Final_Segment_ID', lambda x: x.index.min())
).reset_index().sort_values('first_idx')

ordered_segs = list(zip(seg_order_df['Final_Segment_ID'].tolist(), seg_order_df['state_final'].tolist()))

runs = []
current_run = []
for seg_id, state in ordered_segs:
    if state in (0, 2, 3):
        current_run.append((seg_id, state))
    else:
        if current_run:
            runs.append(current_run)
            current_run = []
        runs.append([(seg_id, state)])
if current_run:
    runs.append(current_run)

merge_ops = {}
for run in runs:
    if len(run) == 1 and run[0][1] == 1:
        continue
    line_positions = [i for i, (_, st) in enumerate(run) if st == 0]
    if not line_positions:
        continue
    target_id = run[line_positions[0]][0]
    for old_id, _ in run:
        if old_id != target_id:
            merge_ops[old_id] = target_id

for old_id, new_id in merge_ops.items():
    df.loc[df['Final_Segment_ID'] == old_id, 'Final_Segment_ID'] = new_id

all_target_ids = set(merge_ops.values())
for tid in all_target_ids:
    df.loc[(df['Final_Segment_ID'] == tid) & (df['State_Final'].isin([2, 3])), 'State_Final'] = 0

alive_seg_ids = sorted(df['Final_Segment_ID'].unique())
id_mapping = {old_id: new_idx for new_idx, old_id in enumerate(alive_seg_ids, start=1)}
df['Final_Segment_ID'] = df['Final_Segment_ID'].map(id_mapping).astype(int)

center_lat = df['Latitude'].mean()
center_lon = df['Longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=16)

all_coords = df[['Latitude', 'Longitude']].values.tolist()
folium.PolyLine(locations=all_coords, color="#000000", weight=5, opacity=0.5).add_to(m)

colors = [
    '#FF0000', '#00FF00', '#0000FF', '#FFFF00', '#FF00FF', '#00FFFF', '#FF8000', '#0080FF', 
    '#FF0080', '#80FF00', '#00FF80', '#FF4040', '#4040FF', '#FFD700', '#ADFF2F', '#FF69B4', 
    '#1E90FF', '#DC143C'
]

sorted_seg_ids = sorted(df['Final_Segment_ID'].unique())

for seg_id in sorted_seg_ids:
    group = df[df['Final_Segment_ID'] == seg_id]
    seg_state_final = group['State_Final'].iloc[0]

    if seg_state_final == 0:
        color = colors[seg_id % len(colors)]
        weight, opacity, dash_array = 3, 0.8, None
    elif seg_state_final == 1:
        color = colors[seg_id % len(colors)]
        weight, opacity, dash_array = 5, 0.9, '5, 5'
    elif seg_state_final == 2:
        color, weight, opacity, dash_array = '#9400D3', 5, 0.9, '10, 5'
    else:
        color, weight, opacity, dash_array = '#FF6600', 5, 0.9, '10, 5'

    indices = group.index.values
    if len(indices) > 0:
        split_locs = np.where(np.diff(indices) > 1)[0] + 1
        for sub_indices in np.split(indices, split_locs):
            if len(sub_indices) < 2: continue
            sub_coords = df.loc[sub_indices, ['Latitude', 'Longitude']].values.tolist()
            folium.PolyLine(
                locations=sub_coords, color=color, weight=weight, 
                opacity=opacity, dash_array=dash_array
            ).add_to(m)

# 지도 출력
m


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start(

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\DrFirst\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start(

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import